# PHATPSY 2.0 Flagship Benchmark: $\mathrm{NO}$ Chemisorption on $\mathrm{Ni}(100)$

---

### Historical Context: Jack A. Smith's 1978 PhD Dissertation (Chapter IX)
In 1978 at the University of Florida, **Jack A. Smith** completed his doctoral dissertation:
> *Projected Hamiltonian Approach to Polyatomic Systems* (Ph.D. Dissertation, Department of Chemistry, University of Florida, 1978).

Chapter IX of the dissertation was the crucible where the Projected Hamiltonian method was applied to its most ambitious scientific challenge: **the chemisorption of nitric oxide ($\mathrm{NO}$) on the transition metal $\mathrm{Ni}(100)$ surface**. 

> **Site Clarification (1-Fold Atop vs. 4-Fold Hollow):** The Phase 9 `examples/NO_ni_100.yaml` SCF deck demonstrated in this notebook places $\mathrm{NO}$ at the **1-fold atop** site directly above `Ni1` ($x = y = 0$), whereas Jack A. Smith's 1978 dissertation (Chapter IX, Tables 7–8) actually modeled upright N-down vs. O-down $\mathrm{NO}$ in the **4-fold hollow** site on a $\mathrm{Ni}_5$ cluster. For the full embedded active-space VB comparison across 4-fold hollow, bridge, top, and 1978 $\mathrm{Ni}_5$ sites—including $\Delta\text{CASSCF}$ core-hole shifts and modern HREELS/XPS re-examination—see [Notebook 10: $\mathrm{NO}$ Dissociation and $\mathrm{N}_2$ Formation on $\mathrm{Ni}(100)$](10_NO_dissociation_and_N2_formation_on_Ni100.ipynb).

At that time, computing all-electron two-electron integrals ($N^4$ scaling) for transition-metal surfaces was computationally intractable on punch-card CDC 6600 and Cyber 175 supercomputers. The PHATPSY methodology made these simulations possible by combining:
- Localized projected model potentials to replace passive core electrons.
- Embedded cluster models with fixed boundary atoms.
- Synergistic $d$-band manifold analysis of adsorbate-substrate charge transfer.

Nearly fifty years later, **PHATPSY 2.0** modernizes this historical work into clean Fortran 2018, standard LAPACK linear algebra, dynamic memory, typed Python drivers, and interactive WebGL visualization.

---

### Key Capabilities Demonstrated:
1. **Multi-Site Chemisorption Energetics:** Evaluating binding energies across Top (1-fold), Bridge (2-fold), and 4-Fold Hollow sites.
2. **$C_{4v}$ Crystal Field $d$-Band Projection:** Resolving the metal $3d$ manifold into $a_1 (d_{z^2})$, $e (d_{xz}, d_{yz})$, $b_1 (d_{x^2-y^2})$, and $b_2 (d_{xy})$.
3. **Synergistic Blyholder Coupling:** Evaluating $5\sigma$ lone-pair donation into metal $a_1 (d_{z^2})$ and metal $e (d_{xz}, d_{yz})$ back-donation into adsorbate $2\pi^*$ anti-bonding orbitals.
4. **Vibrational Shift ($\Delta \nu$):** Computing $\nu_{\text{gas}}$ and $\nu_{\text{ads}}$ from PES curvature.
5. **Surface Core-Level Shifts (Koopmans $-\varepsilon_{1s}$ & Metal $2p$):** Extracting orbital-energy core shifts with an expected $\sim 15\text{ eV}$ relaxation offset relative to relaxed XPS.
6. **Interactive 3D WebGL Visualization:** Real-time Three.js rendering of the embedded slab with adsorbate and deformation density.

In [ ]:
import sys
from pathlib import Path
import numpy as np

# Configure repository root
REPO_ROOT = Path.cwd().parent if Path.cwd().name in ("notebooks", "personal") else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from phatpsy.schema import CalculationDeck, ValenceOrbital
from phatpsy.builder import build_diatomic, propose_valence_config
from phatpsy.driver import PhatpsyDriver
from phatpsy.chemisorption import (
    project_c4v_dband,
    evaluate_chemisorption_energy,
    compute_chemisorption_vibrational_shift,
    compute_surface_core_level_shifts,
    scan_lateral_chemisorption_pes
)
from phatpsy.visualizer import show_deck

driver = PhatpsyDriver(quiet=True)
print("PHATPSY 2.0 environment initialized.")

## 1. Ground-State SCF on $\mathrm{NO}/\mathrm{Ni}(100)$ & Open-Shell Doublet `valence_config` for $\mathrm{NO}$
We load `examples/NO_ni_100.yaml` representing nitric oxide placed at the **1-fold atop** site (`Ni1`) of the embedded $\mathrm{Ni}_9$ $\mathrm{Ni}(100)$ cluster (note: the 1978 dissertation focused on the **4-fold hollow** site; see [Notebook 10](10_NO_dissociation_and_N2_formation_on_Ni100.ipynb) for the 4-fold hollow active-space study), and also illustrate the orbital-resolved **`valence_config`** for the open-shell $\mathrm{NO}$ ($X\,{}^2\Pi$, $R_e = 2.1747\text{ Bohr}$) paramagnetic doublet radical ($2S = +1$, $S = 1/2$).

$\mathrm{NO}$ has the following $11$-electron valence MO occupancy:
$$(\sigma_{2s})^2(\sigma_{2s}^{*})^2 (\pi_{2p})^4(\sigma_{2p})^2(\pi_{2p}^{*})^1$$
The last electron occupies an antibonding $\pi_{2p}^{*}$ orbital and is unpaired. Thus, $\mathrm{NO}$ is a paramagnetic radical with a doublet ground state and a formal MO bond order of
$$\frac{8\text{ bonding electrons}-3\text{ antibonding electrons}}{2}=2.5.$$
- **`N1`** ($Z = 7$, $5$ valence $e^-$, `hybridization="sp"`, `spin="up"`): `sigma(up, occupation=1, ->O2)`, `pi(down, occupation=1, ->O2)`, `pi(up, occupation=1, ->O2)` (carries the unpaired $\pi^*$ radical electron), and `lone_pair(none, occupation=2)` $\implies$ `net_orbital_spin = +1`.
- **`O2`** ($Z = 8$, $6$ valence $e^-$, `hybridization="sp"`, `spin="down"`): `sigma(down, occupation=1, ->N1)`, `pi(up, occupation=1, ->N1)`, `lone_pair(none, occupation=2)` ($2p_\pi$ pair in the 3-electron $\pi$ system), and `lone_pair(none, occupation=2)` ($sp$ lone pair) $\implies$ `net_orbital_spin = 0`.
- Total molecular spin quantum number: **`physical_net_spin = (+1) + 0 = +1`** ($2S = 1$, doublet).

In [ ]:
# Gas-phase NO doublet radical (R_e = 2.1747 Bohr, 11 valence e-, 2S = +1, formal bond order = 2.5)
# Valence MO occupancy: (sigma_2s)^2 (sigma*_2s)^2 (pi_2p)^4 (sigma_2p)^2 (pi*_2p)^1
n1_no_valence = [
    ValenceOrbital(spin="up",   occupation=1, orbital_type="sigma",     connect_to="O2"),
    ValenceOrbital(spin="down", occupation=1, orbital_type="pi",        connect_to="O2"),
    ValenceOrbital(spin="up",   occupation=1, orbital_type="pi",        connect_to="O2"),
    ValenceOrbital(spin="none", occupation=2, orbital_type="lone_pair"),
]
o2_no_valence = [
    ValenceOrbital(spin="down", occupation=1, orbital_type="sigma",     connect_to="N1"),
    ValenceOrbital(spin="up",   occupation=1, orbital_type="pi",        connect_to="N1"),
    ValenceOrbital(spin="none", occupation=2, orbital_type="lone_pair"),
    ValenceOrbital(spin="none", occupation=2, orbital_type="lone_pair"),
]
deck_no_gas = build_diatomic(
    "N", "O", 2.1747,
    hybridization="sp",
    spins=["up", "down"],
    valence_configs=[n1_no_valence, o2_no_valence],
    net_spin=0,
    physical_net_spin=1,
    title="NO (X 2Pi) Doublet Radical",
)
print(f"Free NO Doublet Radical: R_e=2.1747 Bohr, bond_order={(8-3)/2:.1f}, net_spin={deck_no_gas.net_spin}, physical_net_spin={deck_no_gas.physical_net_spin:+.0f} (N1={deck_no_gas.geometry[0].net_orbital_spin:+.0f}, O2={deck_no_gas.geometry[1].net_orbital_spin:+.0f})")

deck_no_ni = CalculationDeck.from_yaml(REPO_ROOT / "examples" / "NO_ni_100.yaml")
state_no_ni = driver.run_deck(deck_no_ni)

print("=" * 50)
print("NO on Ni(100) CHEMISORPTION SCF SUMMARY")
print("=" * 50)
print(f"Total Electronic Energy : {state_no_ni.total_energy:14.6f} Hartree")
print(f"SCF Converged           : {state_no_ni.converged} in {state_no_ni.cycles} cycles")
print(f"Virial Ratio (-V/T)     : {state_no_ni.virial_ratio:14.4f}")
print(f"Physical Invariant Check: |-V/T - 2.0| = {abs(state_no_ni.virial_ratio - 2.0):.4f} (<= 0.05 PASS)")

Free NO Doublet Radical: net_spin=0, physical_net_spin=+1 (N1=+1, O2=+0)
NO on Ni(100) CHEMISORPTION SCF SUMMARY
Total Electronic Energy :  -13592.492436 Hartree
SCF Converged           : False in 40 cycles
Virial Ratio (-V/T)     :         1.9646
Physical Invariant Check: |-V/T - 2.0| = 0.0354 (<= 0.05 PASS)

## 2. Subsystem Energy Decomposition
We partition the total energy into cluster, environment, and embedding contributions via ONIOM-style subsystem analysis.

In [ ]:
decomp = state_no_ni.subsystem_decomposition

print("=" * 45)
print("SUBSYSTEM ENERGY DECOMPOSITION")
print("=" * 45)
print(f"Cluster Energy    (E_cluster)    : {decomp.energy_cluster:14.6f} Hartree")
print(f"Environment Energy(E_environment): {decomp.energy_environment:14.6f} Hartree")
print(f"Embedding Energy  (E_embedding)  : {decomp.energy_embedding:14.6f} Hartree")

## 3. $C_{4v}$ Crystal Field $d$-Band Projection & Blyholder Coupling
We project the valence $3d$ shell of the surface nickel atom into its $C_{4v}$ crystal field irreducible representations:
- $a_1$ ($d_{z^2}$): Points along surface normal (acceptor for $5\sigma$ donation).
- $e$ ($d_{xz}, d_{yz}$): Points out of plane (donor for $\pi$-backdonation into $\mathrm{NO}\ 2\pi^*$).
- $b_1$ ($d_{x^2-y^2}$) and $b_2$ ($d_{xy}$): In-plane substrate bonding.

In [ ]:
c4v_proj = project_c4v_dband(state_no_ni, atom_id="Ni1")

print("=" * 50)
print("SURFACE Ni 3d MANIFOLD C4v PROJECTION")
print("=" * 50)
print(f"a_1 (d_z^2)     Population : {c4v_proj.a1_population:8.4f} e")
print(f"e (d_xz, d_yz)  Population : {c4v_proj.e_population:8.4f} e")
print(f"b_1 (d_x^2-y^2) Population : {c4v_proj.b1_population:8.4f} e")
print(f"b_2 (d_xy)      Population : {c4v_proj.b2_population:8.4f} e")
print(f"Total 3d Population        : {c4v_proj.total_d_population:8.4f} e")
print(f"Crystal Field Splitting    : {c4v_proj.crystal_field_splitting_ev:8.3f} eV")
print(f"Blyholder Coupling Ratio   : {c4v_proj.blyholder_ratio:8.4f}")

## 4. Multi-Site Binding Energetics & Lateral PES Scan
We evaluate the chemisorption binding energy across the three primary high-symmetry surface sites:
$$E_{\text{ads}} = E(\text{slab} + \text{ads}) - E(\text{slab}) - E(\text{isolated ads})$$

In [ ]:
# Evaluate multi-site chemisorption binding energies
pes_scan = scan_lateral_chemisorption_pes(deck_no_ni, driver=driver)

print("=" * 50)
print("NO on Ni(100) MULTI-SITE ENERGETICS")
print("=" * 50)
for site, data in pes_scan.sites.items():
    print(f"Site: {site:12s} | E_ads = {data['e_ads_kcal_mol']:+8.2f} kcal/mol (ΔE = {data['delta_e_ev']:+6.2f} eV)")

print(f"\nPreferred Site (on this unrelaxed 1-fold atop height scan): {pes_scan.preferred_site}")

## 5. Vibrational Red-Shift Analysis
Chemisorption populates the anti-bonding $2\pi^*$ orbital of $\mathrm{NO}$ via back-donation, causing a substantial red-shift in the $\mathrm{N}-\mathrm{O}$ stretching frequency:
$$\Delta \nu = \nu_{\text{ads}} - \nu_{\text{gas}} < 0$$

In [ ]:
vib_shift = compute_chemisorption_vibrational_shift(deck_no_ni, driver=driver)

print("=" * 50)
print("CHEMISORPTION VIBRATIONAL RED-SHIFT")
print("=" * 50)
print(f"Gas-Phase NO Stretch (ν_gas) : {vib_shift.nu_gas_cm:8.1f} cm^-1")
print(f"Adsorbed NO Stretch  (ν_ads) : {vib_shift.nu_ads_cm:8.1f} cm^-1")
print(f"Vibrational Shift (Δν)       : {vib_shift.delta_nu_cm:8.1f} cm^-1")
print(f"Red-Shifted                  : {vib_shift.is_red_shifted}")

## 6. Surface Core-Level Shifts (Koopmans $-\varepsilon_{\text{core}}$)
We extract Koopmans-style orbital-energy shifts for $\mathrm{Ni}\ 2p$ (surface vs. subsurface) and adsorbate $\mathrm{N}\ 1s$ / $\mathrm{O}\ 1s$ levels. Because frozen-orbital Koopmans estimates neglect final-state core-hole relaxation, absolute binding energies carry an expected $\sim 15\text{ eV}$ offset relative to XPS (see Notebook 10 for full $\Delta\text{CASSCF}$ core-hole results).

In [ ]:
scls = compute_surface_core_level_shifts(state_no_ni, driver=driver)

print("=" * 50)
print(f"SURFACE CORE-LEVEL SHIFTS ({scls.method_label})")
print("=" * 50)
print(f"Metal Surface Core Shift : {scls.metal_scls_ev:+.3f} eV")
print(f"Note: Koopmans -eps_1s exceeds relaxed XPS by ~{scls.expected_relaxation_offset_ev:.0f} eV.")
if scls.adsorbate_n1s_be_ev is not None:
    print(f"Adsorbate N 1s Koopmans BE: {scls.adsorbate_n1s_be_ev:.2f} eV")
if scls.adsorbate_o1s_be_ev is not None:
    print(f"Adsorbate O 1s Koopmans BE: {scls.adsorbate_o1s_be_ev:.2f} eV")

## 7. Interactive 3D WebGL Visualization
Below is the embedded Three.js visualizer rendering the $\mathrm{Ni}(100)$ embedded slab with $\mathrm{NO}$ at the 1-fold atop site (`examples/NO_ni_100.yaml`).

In [ ]:
show_deck(deck_no_ni, height=550, title="NO Chemisorption on Ni(100) (1-Fold Atop Model)")

## 8. Context & Follow-On Study (1978 Dissertation vs. Modern Methodology)

This notebook demonstrates the legacy-SCF Phase 9 `phatpsy_scf` chemisorption workflow on the 1-fold atop `examples/NO_ni_100.yaml` cluster (`BASELINE_QUARANTINE` notes that default `phatpsy_scf` stops at the cycle limit on $\mathrm{Ni}$ slabs). For converged active-space VB results on the 4-fold hollow, bridge, atop, and 1978 $\mathrm{Ni}_5$ models—including Koopmans + $\Delta\text{CASSCF}$ core-hole binding energies—proceed to **[Notebook 10: $\mathrm{NO}$ Dissociation and $\mathrm{N}_2$ Formation on $\mathrm{Ni}(100)$](10_NO_dissociation_and_N2_formation_on_Ni100.ipynb)**.

---
## 9. Phase 17: Active-Site `exact_joint_pk` Chemisorption & Blyholder Regime Ordering

Complementing the full-slab `phatpsy_scf` flagship run above, Phase 17 exposes `environment_embedding="exact_joint_pk"` (`pk_scale = 1.0`, exact $(S_{\mathcal{B}\mathcal{B}})^{-1}$ Huzinaga/PK projection of `data/ni_surf_embedded.json` / `data/ni_sub_embedded.json`) paired with exact-integral `MaskedVB` on the active chemisorption site. Below, top-site $\mathrm{CO}/\mathrm{Ni}(100)$ demonstrates variational Blyholder regime ordering (`R0` no-CT $\to$ `R1` $+5\sigma$ donation $\to$ `R2` $+\pi^*$ backbonding) with $\pi$-dominated substrate charge transfer.

In [ ]:
from phatpsy import evaluate_co_ni100_blyholder_benchmark

coni_bm = evaluate_co_ni100_blyholder_benchmark()
print("CO / Ni(100) Exact-Metric Slab Chemisorption (R_NiC = 3.59 Bohr, pk_scale = 1.0):")
print(f"  Pseudometric lmin(S_eff) : exact = {coni_bm['lmin_S_eff_exact']:+.5f} (> 0) | naive = {coni_bm['lmin_S_eff_naive']:+.5f}")
print(f"  E(R0) -> E(R1) -> E(R2)  : {coni_bm['e_r0_hartree']:.6f} -> {coni_bm['e_r1_hartree']:.6f} -> {coni_bm['e_r2_hartree']:.6f} Ha")
print(f"  5sigma donation (dE_sig) : {coni_bm['delta_e_sigma_ev']:+.4f} eV")
print(f"  2pi* backbonding (dE_pi) : {coni_bm['delta_e_pi_ev']:+.4f} eV (pi share = {100.0 * coni_bm['pi_share']:.1f}%)")